**Part 2 · LangChain** · step 15b of 25
[← 15a Runtime context](15a_runtime_context.ipynb) · [🏠 Roadmap](../README.md) · [15c RAG →](15c_rag.ipynb)

# 15b · Streaming Agents (sync + async)

An agent run can take 5–30 seconds (several model calls + tools). With `invoke` the user stares at a
spinner the whole time. With `stream` you show progress **as it happens**: tokens, tool calls, status.

In notebook 04 you streamed a **model**. Here you stream an **agent**, which has several steps, so you
choose *what* to receive with `stream_mode`:

| `stream_mode` | You get after each step… | Use it for |
|---|---|---|
| `"values"` | the **full state** (all messages so far) | simple UIs, debugging |
| `"updates"` | only **what that step changed**, keyed by node (`model`, `tools`) | progress logs ("calling tool X…") |
| `"messages"` | LLM output **token by token**: `(chunk, metadata)` | ChatGPT-style typing effect |
| `"custom"` | anything your tool sends with `runtime.stream_writer` | "Searching… 3/10 done" |

You can pass several: `stream_mode=["updates", "custom"]` → you then get `(mode, chunk)` tuples.

In [ ]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

In [ ]:
import time

from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import ToolRuntime, tool


@tool
def get_weather(city: str, runtime: ToolRuntime) -> str:
    """Get the current weather for a city."""
    runtime.stream_writer(f"🔎 looking up weather for {city}...")  # used by stream_mode="custom"
    time.sleep(1)  # pretend this is a slow API
    runtime.stream_writer(f"✅ got weather for {city}")
    return f"It is 24°C and sunny in {city}."


agent = create_agent(
    model="gpt-4o-mini",
    tools=[get_weather],
    system_prompt="You are a helpful weather assistant. Keep answers to two sentences.",
)

question = {"messages": [HumanMessage("What's the weather in Delhi and Mumbai?")]}

## 1. `stream_mode="values"`: the full state after every step
Each chunk is the whole state, so the newest message is `chunk["messages"][-1]`.

In [ ]:
for chunk in agent.stream(question, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

## 2. `stream_mode="updates"`: only what changed, per node
Each chunk is `{node_name: state_update}`. `create_agent` has two main nodes: `model` and `tools`.
(Middleware hooks show up as extra nodes like `SummarizationMiddleware.before_model`, and their update can be `None`.)

In [ ]:
for chunk in agent.stream(question, stream_mode="updates"):
    for node, update in chunk.items():
        if not update:
            continue
        last = update["messages"][-1]
        if node == "model" and last.tool_calls:
            print(f"[model] wants to call: {[(c['name'], c['args']) for c in last.tool_calls]}")
        elif node == "model":
            print(f"[model] final answer: {last.content}")
        elif node == "tools":
            print(f"[tools] {last.name} returned: {last.content}")

## 3. `stream_mode="messages"`: token-by-token output
Each item is a `(message_chunk, metadata)` tuple. `metadata["langgraph_node"]` tells you which node
produced it. Filter to `"model"` so you only print what the LLM writes, not tool results.

In [ ]:
for token, metadata in agent.stream(question, stream_mode="messages"):
    if metadata["langgraph_node"] == "model" and token.content:
        print(token.content, end="", flush=True)

In [ ]:
# Tool calls also stream in pieces (the JSON arguments arrive a few characters at a time)
for token, metadata in agent.stream(question, stream_mode="messages"):
    for piece in getattr(token, "tool_call_chunks", None) or []:
        print(piece)

## 4. `stream_mode="custom"`: your own progress events
Whatever a tool passes to `runtime.stream_writer(...)` arrives here. Great for long tools
(web search, DB queries, file processing) where the user would otherwise see nothing.

In [ ]:
for event in agent.stream(question, stream_mode="custom"):
    print(event)

## 5. Several modes at once
With a list of modes each item becomes `(mode, chunk)`. This is what a real chat UI does:
tokens for the text, custom events for a status line.

In [ ]:
for mode, chunk in agent.stream(question, stream_mode=["messages", "custom"]):
    if mode == "custom":
        print(f"\n   [status] {chunk}")
    else:
        token, metadata = chunk
        if metadata["langgraph_node"] == "model" and token.content:
            print(token.content, end="", flush=True)

## 6. Async: `ainvoke` and `astream`

Every sync method has an async twin: `invoke` → `ainvoke`, `stream` → `astream`, `batch` → `abatch`.

**Why it matters:** a web server (FastAPI) handles many users on one event loop. A sync `invoke`
blocks that loop for the whole agent run, so other users wait. `await agent.ainvoke(...)` frees the loop while
waiting on the LLM.

Jupyter already runs an event loop, so you can use `await` directly in a cell.
(In a `.py` script, wrap it in `async def main()` and call `asyncio.run(main())`.)

In [ ]:
async for token, metadata in agent.astream(question, stream_mode="messages"):
    if metadata["langgraph_node"] == "model" and token.content:
        print(token.content, end="", flush=True)

In [ ]:
import asyncio

cities = ["Paris", "Tokyo", "New York"]

start = time.perf_counter()
results = await asyncio.gather(*[
    agent.ainvoke({"messages": [HumanMessage(f"Weather in {city}?")]}) for city in cities
])
print(f"3 agent runs concurrently: {time.perf_counter() - start:.1f}s")

for city, r in zip(cities, results):
    print(f"{city:<9} → {r['messages'][-1].content}")

Run sequentially, those three would take about 3× as long. With `gather` they overlap while each waits on the API.

### Streaming to a frontend (FastAPI + Server-Sent Events)
This is the usual production pattern (not run here):

```python
from fastapi import FastAPI
from fastapi.responses import StreamingResponse

app = FastAPI()

@app.get("/chat")
async def chat(q: str, thread_id: str):
    async def event_stream():
        async for token, meta in agent.astream(
            {"messages": [HumanMessage(q)]},
            {"configurable": {"thread_id": thread_id}},
            stream_mode="messages",
        ):
            if meta["langgraph_node"] == "model" and token.content:
                yield f"data: {token.content}\n\n"   # SSE format
        yield "data: [DONE]\n\n"
    return StreamingResponse(event_stream(), media_type="text/event-stream")
```

Streaming works the same with a checkpointer (`thread_id` in the config) and with runtime `context=`.

## Revision notes
- `agent.stream(input, config, stream_mode=...)` yields as the agent runs; `invoke` waits for the end.
- `values` = full state each step · `updates` = per-node diff · `messages` = `(token, metadata)` ·
  `custom` = your `runtime.stream_writer(...)` events.
- Filter tokens with `metadata["langgraph_node"] == "model"`.
- List of modes → `(mode, chunk)` tuples.
- Async twins: `ainvoke`, `astream`, `abatch`. Use them in web servers; `asyncio.gather` runs agents concurrently.
- Frontend: async generator → `StreamingResponse` (SSE) or WebSockets.

---
✅ **Done with 15b Streaming.** Next: [15c RAG →](15c_rag.ipynb)
[🏠 Roadmap](../README.md)